In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


# This works whether the notebook starts from the main
# project folder or from the notebooks folder.
project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

raw_data_folder = project_folder / "data" / "raw"

merchants = pd.read_csv(
    raw_data_folder / "merchants.csv",
    parse_dates=["onboarding_date"]
)

customers = pd.read_csv(
    raw_data_folder / "customers.csv",
    parse_dates=["account_created_at"]
)

payments = pd.read_csv(
    raw_data_folder / "payments.csv",
    parse_dates=["transaction_timestamp"]
)

print("Merchants:", merchants.shape)
print("Customers:", customers.shape)
print("Payments:", payments.shape)

Merchants: (100, 10)
Customers: (10000, 9)
Payments: (50000, 28)


In [2]:
payments.head()


,payment_id,transaction_timestamp,merchant_id,customer_id,amount,currency,processing_fee,payment_method,card_type,card_network,...,country_mismatch_flag,velocity_1h,authentication_used,authentication_result,risk_score,decision,authorisation_status,decline_reason,confirmed_fraud,refund_status
0,P0004693,2025-01-01 00:01:04,M00017,C002316,45.21,AUD,0.83,Card,Debit,Visa,...,0,1,1,Successful,32,Approve,Approved,Not Applicable,0,Not Refunded
1,P0043213,2025-01-01 00:39:48,M00095,C004448,283.54,GBP,4.17,Card,Debit,Mastercard,...,0,1,1,Successful,23,Approve,Approved,Not Applicable,0,Not Refunded
2,P0047874,2025-01-01 00:41:14,M00023,C003680,1022.72,GBP,14.52,Card,Debit,Mastercard,...,0,1,0,Not Attempted,28,Approve,Declined,Issuer Decline,0,Not Refunded
3,P0031576,2025-01-01 00:54:17,M00034,C005766,28.57,USD,0.60,Card,Debit,Visa,...,0,1,1,Successful,49,Approve,Approved,Not Applicable,0,Not Refunded
4,P0017855,2025-01-01 01:16:45,M00069,C007132,330.48,CAD,4.83,Bank Transfer,Not Applicable,Not Applicable,...,0,1,0,Not Attempted,18,Approve,Approved,Not Applicable,0,Not Refunded


In [3]:
payments.info()


<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 28 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   payment_id             50000 non-null  str           
 1   transaction_timestamp  50000 non-null  datetime64[us]
 2   merchant_id            50000 non-null  str           
 3   customer_id            50000 non-null  str           
 4   amount                 50000 non-null  float64       
 5   currency               50000 non-null  str           
 6   processing_fee         50000 non-null  float64       
 7   payment_method         50000 non-null  str           
 8   card_type              50000 non-null  str           
 9   card_network           50000 non-null  str           
 10  issuer_country         50000 non-null  str           
 11  billing_country        50000 non-null  str           
 12  ip_country             50000 non-null  str           
 13  device_type 

In [4]:
required_columns = [
    "payment_id",
    "transaction_timestamp",
    "merchant_id",
    "customer_id",
    "amount",
    "currency",
    "processing_fee",
    "payment_method",
    "card_type",
    "card_network",
    "issuer_country",
    "billing_country",
    "ip_country",
    "device_type",
    "device_id",
    "is_new_device",
    "is_new_customer",
    "is_cross_border",
    "country_mismatch_flag",
    "velocity_1h",
    "authentication_used",
    "authentication_result",
    "risk_score",
    "decision",
    "authorisation_status",
    "decline_reason",
    "confirmed_fraud",
    "refund_status"
]

missing_columns = [
    column
    for column in required_columns
    if column not in payments.columns
]

extra_columns = [
    column
    for column in payments.columns
    if column not in required_columns
]

print("Missing columns:", missing_columns)
print("Unexpected columns:", extra_columns)
print("Column count:", len(payments.columns))

Missing columns: []
Unexpected columns: []
Column count: 28


In [5]:
print("Total missing values:")
print(payments.isna().sum().sum())

print("\nMissing values by column:")
print(payments.isna().sum())

print(
    "\nDuplicate payment IDs:",
    payments["payment_id"].duplicated().sum()
)

print(
    "Duplicate complete rows:",
    payments.duplicated().sum()
)

Total missing values:
0

Missing values by column:
payment_id               0
transaction_timestamp    0
merchant_id              0
customer_id              0
amount                   0
currency                 0
processing_fee           0
payment_method           0
card_type                0
card_network             0
issuer_country           0
billing_country          0
ip_country               0
device_type              0
device_id                0
is_new_device            0
is_new_customer          0
is_cross_border          0
country_mismatch_flag    0
velocity_1h              0
authentication_used      0
authentication_result    0
risk_score               0
decision                 0
authorisation_status     0
decline_reason           0
confirmed_fraud          0
refund_status            0
dtype: int64

Duplicate payment IDs: 0
Duplicate complete rows: 0


In [6]:
invalid_merchant_ids = (
    ~payments["merchant_id"]
    .isin(merchants["merchant_id"])
).sum()

invalid_customer_ids = (
    ~payments["customer_id"]
    .isin(customers["customer_id"])
).sum()

print("Invalid merchant IDs:", invalid_merchant_ids)
print("Invalid customer IDs:", invalid_customer_ids)

Invalid merchant IDs: 0
Invalid customer IDs: 0


In [7]:
print(
    "First transaction:",
    payments["transaction_timestamp"].min()
)

print(
    "Last transaction:",
    payments["transaction_timestamp"].max()
)

print(
    "Transactions outside 2025:",
    (
        ~payments["transaction_timestamp"].between(
            "2025-01-01 00:00:00",
            "2025-12-31 23:59:59"
        )
    ).sum()
)

print(
    "Zero or negative amounts:",
    (payments["amount"] <= 0).sum()
)

print(
    "Negative processing fees:",
    (payments["processing_fee"] < 0).sum()
)

expected_fees = (
    payments["amount"] * 0.014 + 0.20
).round(2)

print(
    "Incorrect processing fees:",
    (
        ~np.isclose(
            payments["processing_fee"],
            expected_fees
        )
    ).sum()
)

First transaction: 2025-01-01 00:01:04
Last transaction: 2025-12-31 23:54:04
Transactions outside 2025: 0
Zero or negative amounts: 0
Negative processing fees: 0
Incorrect processing fees: 37


In [8]:
binary_columns = [
    "is_new_device",
    "is_new_customer",
    "is_cross_border",
    "country_mismatch_flag",
    "authentication_used",
    "confirmed_fraud"
]

for column in binary_columns:
    invalid_values = (
        ~payments[column].isin([0, 1])
    ).sum()

    print(column, "invalid values:", invalid_values)

print(
    "Risk scores outside 0–100:",
    (
        ~payments["risk_score"].between(0, 100)
    ).sum()
)

print(
    "Velocity below 1:",
    (payments["velocity_1h"] < 1).sum()
)

is_new_device invalid values: 0
is_new_customer invalid values: 0
is_cross_border invalid values: 0
country_mismatch_flag invalid values: 0
authentication_used invalid values: 0
confirmed_fraud invalid values: 0
Risk scores outside 0–100: 0
Velocity below 1: 0


In [9]:
category_columns = [
    "currency",
    "payment_method",
    "card_type",
    "card_network",
    "device_type",
    "authentication_result",
    "decision",
    "authorisation_status",
    "decline_reason",
    "refund_status"
]

for column in category_columns:
    print(f"\n{column}:")
    print(sorted(payments[column].unique()))


currency:
['AUD', 'CAD', 'EUR', 'GBP', 'USD']

payment_method:
['Bank Transfer', 'Card', 'Digital Wallet']

card_type:
['Credit', 'Debit', 'Not Applicable']

card_network:
['Amex', 'Mastercard', 'Not Applicable', 'Visa']

device_type:
['Desktop', 'Mobile', 'Tablet']

authentication_result:
['Failed', 'Not Attempted', 'Successful']

decision:
['Approve', 'Block', 'Review']

authorisation_status:
['Approved', 'Declined']

decline_reason:
['Authentication Failed', 'Insufficient Funds', 'Invalid Details', 'Issuer Decline', 'Not Applicable', 'Risk Block', 'Velocity Limit']

refund_status:
['Fully Refunded', 'Not Refunded', 'Partially Refunded']


In [10]:
bank_transfer_errors = payments[
    (payments["payment_method"] == "Bank Transfer")
    & (
        (payments["card_type"] != "Not Applicable")
        | (
            payments["card_network"]
            != "Not Applicable"
        )
    )
]

authentication_errors = payments[
    (
        (payments["authentication_used"] == 0)
        & (
            payments["authentication_result"]
            != "Not Attempted"
        )
    )
    | (
        (payments["authentication_used"] == 1)
        & (
            payments["authentication_result"]
            == "Not Attempted"
        )
    )
]

approved_decline_errors = payments[
    (payments["authorisation_status"] == "Approved")
    & (
        payments["decline_reason"]
        != "Not Applicable"
    )
]

declined_reason_errors = payments[
    (payments["authorisation_status"] == "Declined")
    & (
        payments["decline_reason"]
        == "Not Applicable"
    )
]

fraud_approval_errors = payments[
    (payments["confirmed_fraud"] == 1)
    & (
        payments["authorisation_status"]
        != "Approved"
    )
]

print(
    "Bank-transfer card errors:",
    len(bank_transfer_errors)
)

print(
    "Authentication consistency errors:",
    len(authentication_errors)
)

print(
    "Approved-payment decline errors:",
    len(approved_decline_errors)
)

print(
    "Declined payments without reasons:",
    len(declined_reason_errors)
)

print(
    "Fraud recorded on unapproved payments:",
    len(fraud_approval_errors)
)

Bank-transfer card errors: 0
Authentication consistency errors: 0
Approved-payment decline errors: 0
Declined payments without reasons: 0
Fraud recorded on unapproved payments: 0


In [11]:
customer_dates = customers[
    ["customer_id", "account_created_at"]
]

payment_customer_dates = payments.merge(
    customer_dates,
    on="customer_id",
    how="left"
)

payments_before_account = payment_customer_dates[
    payment_customer_dates["transaction_timestamp"]
    < payment_customer_dates["account_created_at"]
]

print(
    "Payments before account creation:",
    len(payments_before_account)
)

Payments before account creation: 0


In [12]:
payments["month"] = (
    payments["transaction_timestamp"]
    .dt.to_period("M")
)

monthly_summary = (
    payments.groupby("month")
    .agg(
        attempts=("payment_id", "count"),
        total_amount=("amount", "sum"),
        approval_rate=(
            "authorisation_status",
            lambda values: (
                values == "Approved"
            ).mean()
        ),
        fraud_rate=("confirmed_fraud", "mean")
    )
)

monthly_summary.round(4)


,attempts,total_amount,approval_rate,fraud_rate
month,,,,
2025-01,2167,300711.48,0.8976,0.0046
2025-02,2068,292855.13,0.9038,0.0029
2025-03,2512,335981.43,0.8913,0.0076
2025-04,2746,357480.27,0.8980,0.0058
2025-05,3243,453641.35,0.8915,0.0062
2025-06,3814,623871.50,0.8983,0.0047
2025-07,4154,657300.53,0.8987,0.0072
2025-08,4420,711981.74,0.8919,0.0054
2025-09,3873,536692.83,0.8854,0.0059


In [13]:
merchant_24 = payments[
    payments["merchant_id"] == "M00024"
].copy()

merchant_24["period"] = np.where(
    merchant_24["transaction_timestamp"]
    < pd.Timestamp("2025-10-01"),
    "Jan-Sep",
    "Oct-Dec"
)

merchant_24_summary = (
    merchant_24.groupby("period")
    .agg(
        attempts=("payment_id", "count"),
        approval_rate=(
            "authorisation_status",
            lambda values: (
                values == "Approved"
            ).mean()
        ),
        authentication_failure_rate=(
            "authentication_result",
            lambda values: (
                values == "Failed"
            ).mean()
        ),
        fraud_rate=("confirmed_fraud", "mean")
    )
)

merchant_24_summary.round(4)

,attempts,approval_rate,authentication_failure_rate,fraud_rate
period,,,,
Jan-Sep,114,0.8772,0.0789,0.0
Oct-Dec,93,0.7527,0.1828,0.0


In [ ]:
assert payments.shape == (50_000, 29)
# There are temporarily 29 columns because the notebook added "month".

assert payments["payment_id"].is_unique
assert payments.isna().sum().sum() == 0
assert payments["amount"].gt(0).all()
assert payments["processing_fee"].ge(0).all()
assert payments["risk_score"].between(0, 100).all()
assert payments["velocity_1h"].ge(1).all()

assert payments["merchant_id"].isin(
    merchants["merchant_id"]
).all()

assert payments["customer_id"].isin(
    customers["customer_id"]
).all()

assert len(bank_transfer_errors) == 0
assert len(authentication_errors) == 0
assert len(approved_decline_errors) == 0
assert len(declined_reason_errors) == 0
assert len(fraud_approval_errors) == 0
assert len(payments_before_account) == 0

print("STEP 20 PASSED: payment data is valid.")

In [ ]:
assert payments.shape == (50_000, 29)
# There are temporarily 29 columns because the notebook added "month".

assert payments["payment_id"].is_unique
assert payments.isna().sum().sum() == 0
assert payments["amount"].gt(0).all()
assert payments["processing_fee"].ge(0).all()
assert payments["risk_score"].between(0, 100).all()
assert payments["velocity_1h"].ge(1).all()

assert payments["merchant_id"].isin(
    merchants["merchant_id"]
).all()

assert payments["customer_id"].isin(
    customers["customer_id"]
).all()

assert len(bank_transfer_errors) == 0
assert len(authentication_errors) == 0
assert len(approved_decline_errors) == 0
assert len(declined_reason_errors) == 0
assert len(fraud_approval_errors) == 0
assert len(payments_before_account) == 0


STEP 20 PASSED: payment data is valid.
